In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.ensemble import RandomForestClassifier

data = pd.read_csv("ecommerce_churn.csv")


In [ ]:
#Have a look
print(data.head())
print(data.shape)
data.info()
print("Missing values:")
print(data.isnull().sum())

print("Churn counts:")
print(data["Churned"].value_counts())
print("\nChurn percentage:")
print(data["Churned"].value_counts(normalize=True) * 100)

#Split off response variable
X=data.drop(columns=["Churned"])
y=data["Churned"]

In [ ]:
# First split, save 20% for final testing
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=27,
    stratify=y
)

# Second split, split the remaining data into training and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=27,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

In [ ]:
# Check missing values in the training data
print(X_train.isnull().sum())

In [ ]:
# Find numerical columns
numeric_cols = X_train.select_dtypes(include="number").columns

# Calculate medians using training data
train_medians = X_train[numeric_cols].median()

# Fill missing values using the same training medians
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

X_train[numeric_cols] = X_train[numeric_cols].fillna(train_medians)
X_val[numeric_cols] = X_val[numeric_cols].fillna(train_medians)
X_test[numeric_cols] = X_test[numeric_cols].fillna(train_medians)

print("Training missing values:", X_train.isnull().sum().sum())
print("Validation missing values:", X_val.isnull().sum().sum())
print("Testing missing values:", X_test.isnull().sum().sum())

In [ ]:
# Make dummy variables
X_train = pd.get_dummies(X_train, dtype=int)
X_val = pd.get_dummies(X_val, dtype=int)

print(X_train.columns.equals(X_val.columns))

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

In [ ]:
# Scale numerical features using training data
scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])

In [ ]:
# Train a log regression model to predict churn
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predict churn for customers in the test set
y_pred = model.predict(X_val)

# Check accuracy
print("Model accuracy:", model.score(X_val, y_val))

print(confusion_matrix(y_val, y_pred))

In [ ]:
print("Logistic Regression training accuracy:", model.score(X_train, y_train))
print("Logistic Regression validation accuracy:", model.score(X_val, y_val))

In [ ]:
decision_tree=DecisionTreeClassifier(
    max_depth=5,
    random_state=27
)

decision_tree.fit(X_train, y_train)

decision_tree_pred = decision_tree.predict(X_val)

print("Decision tree accuracy:", decision_tree.score(X_val, y_val))
print(confusion_matrix(y_val, decision_tree_pred))

print("Tree training accuracy:", decision_tree.score(X_train, y_train))
print("Tree validation accuracy:", decision_tree.score(X_val, y_val))

# Find the most important features in the tree
feature_importance = pd.Series(
    decision_tree.feature_importances_,
    index=X_train.columns
)

print(feature_importance.sort_values(ascending=False).head(10))


In [ ]:
plt.figure(figsize=(18, 8))

plot_tree(
    decision_tree,
    feature_names=X_train.columns,
    class_names=["Stayed", "Churned"],
    filled=True,
    rounded=True,
    max_depth=2,
    fontsize=9
)

plt.show()

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators= 100, 
    max_depth=15,
    random_state=27
)

random_forest.fit(X_train, y_train)

random_forest_pred = random_forest.predict(X_val)


print("Random Forest accuracy:", random_forest.score(X_val, y_val))
print("Random Forest confusion matrix:")
print(confusion_matrix(y_val, random_forest_pred))

print("Forest training accuracy:", random_forest.score(X_train, y_train))

In [ ]:
# Final test on unused test data
X_test = pd.get_dummies(X_test, dtype=int)

# Check that the sets match
print("Columns match:", X_train.columns.equals(X_test.columns))

In [ ]:
# Apply the scaling used in training
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Final test 
test_pred = random_forest.predict(X_test)

print("Final test accuracy:", random_forest.score(X_test, y_test))
print("Final test confusion matrix:")
print(confusion_matrix(y_test, test_pred))

In [ ]:
# Check which features are considered most important
feature_importance = pd.Series(
    random_forest.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(feature_importance.head(10))

In [ ]:
# Predict churn probabilities for the validation set
churn_probabilities = random_forest.predict_proba(X_val)[:, 1]

# show predictions
results = pd.DataFrame({
    "Actual_Churn": y_val,
    "Churn_Probability": churn_probabilities
})

# Classify customers based on risk
results["Churn_Risk_Level"] = pd.cut(
    results["Churn_Probability"],
    bins=[-0.01, 0.25, 0.75, 1.0],
    labels=["Low", "Medium", "High"]
)


results.head(10)

In [ ]:
#add back unscaled value for alert levels
results = results.join(data[["Lifetime_Value"]])

results["Value_Level"] = pd.cut(
    results["Lifetime_Value"],
    bins=[-0.01, 1851.20, float("inf")],
    labels=["Low", "High"]
)

In [ ]:
results.describe()

In [ ]:
high_risk_alerts = results[
    results["Risk_Level"] == "High"
].copy()

# Sort customers from highest to lowest churn probability
high_risk_alerts = high_risk_alerts.sort_values(
    by="Churn_Probability",
    ascending=False
)

print("High-risk customers:", len(high_risk_alerts))

high_risk_alerts.head(10)

In [ ]:
results["Alert"] = "No alert"

results.loc[
    results["Churn_Risk_Level"] == "High",
    "Alert"
] = "Standard alert"

results.loc[
    (results["Churn_Risk_Level"] == "High") &
    (results["Value_Level"] == "High"),
    "Alert"
] = "Priority alert"

In [ ]:
results["Alert"].value_counts()

In [ ]:
priority_alerts = results[
    results["Alert"] == "Priority alert"
].sort_values("Churn_Probability", ascending=False)

print("Priority alerts:", len(priority_alerts))
display(priority_alerts)

In [ ]:
standard_alerts = results[
    results["Alert"] == "Standard alert"
].sort_values("Churn_Probability", ascending=False)

print("Standard alerts:", len(standard_alerts))
display(standard_alerts)

In [ ]:
import boto3

s3 = boto3.client("s3", region_name="ap-southeast-2")
bucket = "as3-churn-project-bucket"

s3.head_bucket(Bucket=bucket)

print("S3 Bucket Connected")

In [ ]:
results.to_csv("churn_alerts.csv", index=True)

# Upload the report to S3
s3.upload_file(
    "churn_alerts.csv",
    bucket,
    "Reports/churn_alerts.csv"
)

print("Report uploaded to S3")

In [ ]:
model_package = {
    "model": random_forest,
    "scaler": scaler,
    "medians": train_medians,
    "numeric_columns": list(numeric_cols),
    "feature_columns": list(X_train.columns)
}

with open("churn_model.pkl", "wb") as file:
    pickle.dump(model_package, file)

s3.upload_file(
    "churn_model.pkl",
    bucket,
    "Models/churn_model.pkl"
)

print("Model saved to S3")

In [ ]:
s3.download_file(
    bucket,
    "Models/churn_model.pkl",
    "loaded_churn_model.pkl"
)

with open("loaded_churn_model.pkl", "rb") as file:
    saved_package = pickle.load(file)

saved_model = saved_package["model"]

print("Model loaded")
print("Number of trees:", saved_model.n_estimators)

In [ ]:
# Make predictions using the model loaded from S3
loaded_predictions = saved_model.predict_proba(X_val)[:, 1]

# Compare with the original predictions
print("Predictions match:",
      (loaded_predictions == churn_probabilities).all())

In [ ]:
# Take 100 customer records to simulate a new batch
new_customers = data.drop(columns=["Churned"]).sample(
    n=100,
    random_state=27
)

# Save them as a CSV
new_customers.to_csv("new_customers.csv", index=False)

# Upload to S3
s3.upload_file(
    "new_customers.csv",
    bucket,
    "Data/new_customers.csv"
)

print("Test customer data uploaded")
print("Customers:", len(new_customers))

In [ ]:
# Download the customer data from S3
s3.download_file(
    bucket,
    "Data/new_customers.csv",
    "incoming_customers.csv"
)

# Read the downloaded file
incoming_customers = pd.read_csv("incoming_customers.csv")

print("Customers loaded:", len(incoming_customers))
display(incoming_customers.head())

In [ ]:
# Retrieve the preprocessing information saved with our model
numeric_columns = saved_package["numeric_columns"]
feature_columns = saved_package["feature_columns"]
medians = saved_package["medians"]
saved_scaler = saved_package["scaler"]

# Make a copy of the incoming data
new_features = incoming_customers.copy()

# Fill missing numeric values using the training medians
new_features[numeric_columns] = new_features[numeric_columns].fillna(medians)

# Convert categorical columns into numeric dummy variables
new_features = pd.get_dummies(new_features, dtype=int)

# Ensure the columns match the training data
new_features = new_features.reindex(
    columns=feature_columns,
    fill_value=0
)

# Apply the original training scaler
new_features[numeric_columns] = saved_scaler.transform(
    new_features[numeric_columns]
)

print("Customer data prepared")
print("Shape:", new_features.shape)

In [ ]:
# Predict churn probabilities using our saved model
new_probabilities = saved_model.predict_proba(new_features)[:, 1]

# Create a results table
new_results = incoming_customers[["Lifetime_Value"]].copy()

new_results["Churn_Probability"] = new_probabilities

# Assign churn risk levels
new_results["Risk_Level"] = pd.cut(
    new_results["Churn_Probability"],
    bins=[-0.01, 0.25, 0.75, 1.0],
    labels=["Low", "Medium", "High"]
)

# Display our predictions
display(new_results.head(10))

In [ ]:
# Classify customers by lifetime value
new_results["Value_Level"] = pd.cut(
    new_results["Lifetime_Value"],
    bins=[-0.01, 1851.20, float("inf")],
    labels=["Low", "High"]
)

# Create alerts
new_results["Alert"] = "No alert"

new_results.loc[
    new_results["Risk_Level"] == "High",
    "Alert"
] = "Standard alert"

new_results.loc[
    (new_results["Risk_Level"] == "High") &
    (new_results["Value_Level"] == "High"),
    "Alert"
] = "Priority alert"

# Display alert counts
print(new_results["Alert"].value_counts())

# Show customers requiring attention
display(
    new_results[new_results["Alert"] != "No alert"]
    .sort_values("Churn_Probability", ascending=False)
)

In [ ]:
# Save the new prediction results
new_results.to_csv("new_customer_alerts.csv", index=False)

# Upload the report to S3
s3.upload_file(
    "new_customer_alerts.csv",
    bucket,
    "Reports/new_customer_alerts.csv"
)

print("New customer alert report uploaded")